# Agentic AI: From Chatbots to Agents
**Presented by:** Sharad Rajore | **Organization:** Zensar Technologies

---

### 🎯 Learning Objectives
1. **Witness the Limitations:** See where a raw LLM fails (Math & Real-time data).
2. **Build Tools:** Create "Hands" for the AI using Python functions.
3. **The ReAct Loop:** Observe the *Thought -> Action -> Observation* cycle in real-time.

### 🛠️ Prerequisites
- Python 3.10+
- LangChain installed (`pip install langchain langchain-openai`)
- OpenAI API Key (or similar)

In [1]:
# 1. Setup Environment
from dotenv import load_dotenv
load_dotenv()

True

## Part 1: The "Brain in a Jar" (LLM Limitations)
First, let's look at a standard LLM. It is powerful but isolated. It cannot access the outside world and often hallucinates on complex math.

In [2]:
#from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama

# Initialize the "Brain" (LLM)
llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)

# TEST 1: The Math Trap
# LLMs predict the next word, they don't calculate. 
# For large numbers, they often guess.
query = "What is 49,823 multiplied by 19,283?"

response = llm.invoke(query)
print(f"User: {query}")
print(f"LLM Answer: {response.content}")

# Reality Check: 49823 * 19283 = 960,736,909
# Check if the LLM got it right (It often fails or is slightly off without a tool).

User: What is 49,823 multiplied by 19,283?
LLM Answer: The product is  

\[
49{,}823 \times 19{,}283 = 960{,}736{,}909.
\]


## Part 2: Giving the Brain "Hands" (Defining Tools)
To fix this, we don't train the model more. We give it a **Tool**. 
In LangChain, a tool is just a Python function with a `@tool` decorator.

In [3]:
from langchain_core.tools import tool

# 1. Define the Calculator Tool
@tool
def multiply(a: int, b: int) -> int:
    """ Multiplies two integers and returns the result."""
    return a * b

# 2. Define a "Mock" Search Tool (Simulating Real-time data)
@tool
def get_stock_price(ticker: str) -> str:
    """Returns the current stock price for a given ticker symbol. 
    Use 'ZENSAR' for Zensar Technologies, 'GOOGL' for Google."""
    # In a real agent, this would call Yahoo Finance API
    ticker_upper = ticker.upper().strip()
    if "ZENSAR" in ticker_upper:
        return "464.00 INR"
    elif "GOOGL" in ticker_upper or "GOOGLE" in ticker_upper:
        return "175.00 USD"
    else:
        return f"Unknown Ticker: {ticker}"

# List of tools available to our Agent
#tools = [multiply, get_stock_price]

print("Tools Created: Calculator (Hands) & Stock Search (Eyes)")

Tools Created: Calculator (Hands) & Stock Search (Eyes)


In [4]:
from langchain_community.tools import DuckDuckGoSearchResults

search_tool = DuckDuckGoSearchResults()

C:\Users\manda\AppData\Local\Temp\ipykernel_17628\2437116088.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchResults


In [6]:
response = search_tool.invoke("what about US and Iran deal? is it finally done?")

In [7]:
response

'snippet: 3 days ago - As part of the peace proposals, Iran proposed building at least 19 additional reactors, suggesting that American involvement in these projects could help revitalize the US nuclear industry. A planned address by Araghchi to formally announce this proposal was ultimately cancelled. On May 27, Trump said both sides were close to finalizing an agreement involving strong inspections. Araghchi stated he was unsure whether a deal ..., title: 2025–2026 Iran–United States negotiations - Wikipedia, link: https://en.wikipedia.org/wiki/2025–2026_Iran–United_States_negotiations, snippet: 3 weeks ago - They will likely have to contend with the aftermath without help from the United States. According to the MOU, U.S. forces will withdraw from areas around Iran within thirty days of a final agreement., title: The Iran Deal Reopens the Strait. Much Remains to Be Done. | Council on Foreign Relations, link: https://www.cfr.org/articles/trumps-iran-deal-reopens-the-strait-much-remai

## Part 3: The Agent (ReAct Loop)
Now we bind the tools to the LLM. The LLM will now act as a router:
1. **Thought:** Do I know the answer? No.
2. **Action:** Pick the right tool.
3. **Observation:** Get the output.
4. **Final Answer:** Summarize for the user.

In [6]:
#from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama

# Initialize the "Brain" (LLM)
llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)

In [ ]:
from langchain.agents import create_agent

# Create a ReAct agent using the modern LangChain API
# debug=True enables verbose output showing the Thought -> Action -> Observation loop
#agent_executor = create_agent(llm, tools, debug=True)
agent_executor = create_agent(
    model=llm,
    tools=[multiply, get_stock_price,search_tool,],
    system_prompt="You are a helpful assistant you only provide the answers based on the tools ,don't use your own knowledge and don't make up any answer if you don't have the answer in the tools, if you don't have the answer in the tools then say I don't know",
)


print("Agent created successfully with ReAct capabilities!")

Agent created successfully with ReAct capabilities!


## Part 4: Agent in Action
Let's ask the exact same math question, plus a question about Zensar's stock.

In [9]:
print("--- SCENARIO 1: MATH ---")
result1 = agent_executor.invoke({"messages": [("user", "What is 49,823 multiplied by 19,283?")]})
print(f"\nFinal Answer: {result1['messages'][-1].content}")
#result1



--- SCENARIO 1: MATH ---

Final Answer: The product of 49,823 and 19,283 is 960,736,909.


In [10]:
print("\n\n--- SCENARIO 2: REAL-TIME DATA ---")
result2 = agent_executor.invoke({"messages": [("user", "What is the current stock price of Zensar?")]})
print(f"\nFinal Answer: {result2['messages'][-1].content}")



--- SCENARIO 2: REAL-TIME DATA ---

Final Answer: The current stock price of Zensar Technologies is **₹464.00**.


In [11]:



print("\n\n--- SCENARIO 3: REAL-TIME DATA ---")
result2 = agent_executor.invoke({"messages": [("user", "What is the current stock price of Zensar then after that could you help me with What is 49823 * 19283? then tell me the capital of India")]})
#print(f"\nFinal Answer: {result2['messages'][-1].content}")
result2



--- SCENARIO 3: REAL-TIME DATA ---


{'messages': [HumanMessage(content='What is the current stock price of Zensar then after that could you help me with What is 49823 * 19283? then tell me the capital of India', additional_kwargs={}, response_metadata={}, id='2ada766c-68dd-45d9-8e62-8f0dcec687ce'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-07-06T14:12:04.276024966Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2416958424, 'load_duration': None, 'prompt_eval_count': 321, 'prompt_eval_duration': None, 'eval_count': 224, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'}, id='lc_run--019f37c5-88da-7573-906e-3065d2559e7e-0', tool_calls=[{'name': 'get_stock_price', 'args': {'ticker': 'ZENSAR'}, 'id': 'dabb0d15-0e83-498f-94c1-da3e993f0f10', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 321, 'output_tokens': 224, 'total_tokens': 545}),
  ToolMessage(content='464.00 INR', 

### 🧠 What just happened?
1. **Math:** Instead of guessing, you should see the agent say `Invoking: multiply`. It used the calculator.
2. **Stock:** Instead of saying "I don't know", it said `Invoking: get_stock_price`. 

**This is the shift from Generative AI (guessing) to Agentic AI (executing).**

In [12]:
print("\n\n--- SCENARIO 4: REAL-TIME DATA ---")
result2 = agent_executor.invoke({"messages": [("user", "In India vs Afghanistan cricket match yesterday 17th June 2026, how many runs India has scored?")]})
#print(f"\nFinal Answer: {result2['messages'][-1].content}")
result2



--- SCENARIO 4: REAL-TIME DATA ---


{'messages': [HumanMessage(content='In India vs Afghanistan cricket match yesterday 17th June 2026, how many runs India has scored?', additional_kwargs={}, response_metadata={}, id='2c06b0f4-1b11-4426-855c-572d8870054b'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-07-06T14:13:34.181418938Z', 'done': True, 'done_reason': 'stop', 'total_duration': 938305236, 'load_duration': None, 'prompt_eval_count': 308, 'prompt_eval_duration': None, 'eval_count': 67, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'}, id='lc_run--019f37c6-f0eb-7b71-9d89-441973189dbd-0', tool_calls=[{'name': 'duckduckgo_results_json', 'args': {'query': 'India vs Afghanistan cricket match 17 June 2026 runs India scored'}, 'id': 'dc9482c0-0cf6-49cb-98ce-91e63624e1e2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 308, 'output_tokens': 67, 'total_tokens': 375}),
  ToolMessage(c

In [13]:
from tavily import TavilyClient

tavily_client = TavilyClient(api_key="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")
response = tavily_client.search("Who is Leo Messi?")

print(response)

{'query': 'Who is Leo Messi?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.imdb.com/name/nm2177779', 'title': 'Lionel Messi - IMDb', 'content': "# Lionel Messi. Lionel Messi is a football player from Argentina who plays for Inter Miami. He has won the Ballon D'Or, the annual award given to the best player in the world, 8 times, 2022 FIFA World Cup winner and an Olympic gold medal winner in 2008. He showed an enormous aptitude for football and was in the youth teams for Newell's Old Boys, his local team. Faced with mounting medical expenses to treat a growth hormone condition, Messi's family accepted an offer to move the 13-year-old prodigy to FC Barcelona, who would pay for his treatment. Messi has gone on to become one of the most decorated players in football history and has broken countless records for his club and his country. Image 8: View PosterImage 9: View PosterImage 10: View Poster+ 5 Image 11: View Poster. *   Image 16: Lionel 